# Dreamer — Learning to Act by Imagining

In the [MBRL notebook](./15_MBRL.ipynb), we saw how **Dyna-style** algorithms like MBPO use a
learned dynamics model to generate synthetic transitions in _state space_, effectively augmenting
the replay buffer for a model-free agent (SAC). Dreamer takes a fundamentally different approach:
instead of generating data for a model-free learner, it learns a **latent world model** and then
**imagines entire trajectories** purely in latent space — never decoding back to observations. The
actor and critic are trained entirely on these imagined rollouts, with gradients flowing backward
through the learned dynamics.

This idea was introduced in
[Dream to Control (Hafner et al., 2020)](https://arxiv.org/abs/1912.01603) and refined in
[DreamerV2 (2021)](https://arxiv.org/abs/2010.02193) and
[DreamerV3 (2023)](https://arxiv.org/abs/2301.04104). The core loop is:

1. **Collect** real experience from the environment
2. **Learn** a latent world model (RSSM) from that experience
3. **Imagine** future trajectories in latent space using the learned model
4. **Train** actor and critic on imagined trajectories (no environment calls)
5. Repeat

---

## What you will learn

- What an **RSSM** (Recurrent State-Space Model) is and why it combines deterministic + stochastic
  state.
- How to train a **latent world model** with reconstruction, reward prediction, continuation, and KL
  regularization.
- How **imagination rollouts** work: unrolling the model forward without any real observations.
- How to compute **TD($\lambda$) returns** over imagined trajectories.
- How **gradients flow through the world model** into the actor (the key insight of Dreamer).
- How **continuation modeling** replaces the fixed discount with a learned, state-dependent
  discount.

> **Simplifications vs. full DreamerV3:**
>
> - **Vector observations** (MuJoCo state) instead of pixels.
> - **Gaussian latents** instead of discrete categorical latents.
> - **Smaller networks** and shorter training budgets.
>
> We do include several important DreamerV3 techniques: **two-hot distributional heads** (symlog
> bins for reward and value), **KL balancing**, **return scaling**, and **advantage-based actor
> updates**. The structure is faithful to modern Dreamer agents: RSSM world model + imagined
> rollouts + TD($\lambda$) returns.

---

## Notes on compute

Dreamer-like agents can be compute-hungry. This notebook aims to stay small and readable while
producing a working demo (learning curves should improve). If you have a GPU, it will help.


## The Dreamer Training Loop

At a high level, Dreamer alternates between three processes:

```mermaid
flowchart LR
    subgraph COLLECT["1. COLLECT"]
        ENV["Environment<br/>obs, act, rew, done"]
    end

    subgraph WM["2. WORLD MODEL"]
        REPLAY["Replay Buffer"] --> RSSM["Encoder + RSSM<br/>Decoder, Rew/Cont"]
    end

    subgraph BEHAVIOR["3. BEHAVIOR"]
        IMAGINE["Imagination<br/>(prior only)"] --> AC["Actor + Critic<br/>TD(&lambda;) returns"]
    end

    ENV -- "store" --> REPLAY
    RSSM -- "imagine" --> IMAGINE
    AC -- "policy" --> ENV
```

The key insight: during **step 3**, the actor's gradients flow _backward through the world model
dynamics_ — the model is not just a data generator (as in MBPO), it is a differentiable simulator.


## Setup

We use `gymnasium` (MuJoCo) for the environment and `torch` for all neural network components. No
external RL libraries are needed — everything is implemented from scratch in this notebook.


In [ ]:
# If you run this in a clean environment, you may need:
#   pip install gymnasium[mujoco] torch numpy matplotlib

# Required for torch.use_deterministic_algorithms(True) with cuBLAS GEMM kernels
# (init_random sets deterministic mode). Must be set BEFORE importing torch.
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import math
import time
import random
from dataclasses import dataclass
from typing import Dict, Tuple

import numpy as np

import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

import matplotlib.pyplot as plt

from util.gymnastics import init_random, DEVICE

# Reproducibility — `init_random` seeds python/numpy/torch consistently with the
# rest of the course (see util/gymnastics.py).
SEED = 0
init_random(seed=SEED)

print("Device:", DEVICE)

## Environment

We validate Dreamer on
[InvertedPendulum-v5](https://gymnasium.farama.org/environments/mujoco/inverted_pendulum/), a small
MuJoCo task with a very clean success criterion. A cart runs along a track with a pole hinged on
top; the agent applies a 1-D horizontal force to the cart and must keep the pole upright.
Observations are 4-D (cart position, pole angle, cart velocity, pole angular velocity), the action
is a 1-D force in $[-3, 3]$, and the agent receives a reward of $+1$ per step **while the pole is
upright**. The episode terminates immediately when the pole tips past $\approx 0.2$ rad, and is
time-limited to 1000 steps.

This is the right sanity check for an educational Dreamer: unlike classic-control `Pendulum-v1`
(which never terminates), InvertedPendulum has a **real termination signal** that exercises the
continuation head, and the "solved" bar is unambiguous — sustained eval return of 1000 per episode
means the algorithm is working. A random policy gets roughly 5–20 reward per episode; a good policy
gets 1000.

After InvertedPendulum learns cleanly, we include an optional **Hopper-v5** training cell at the
bottom of the notebook. Hopper is the classic Dreamer benchmark — much harder, much more fragile,
and discussed in detail in the **Traps and Fragility** section. We recommend running
InvertedPendulum first to confirm the algorithm, then trying Hopper as a stretch exercise.


In [ ]:
# Default env: InvertedPendulum-v5. Small MuJoCo task — perfect for verifying
# the Dreamer algorithm learns cleanly. Dense reward (+1 per upright step),
# real termination when the pole falls, 1000-step time limit. Solved = 1000.
ENV_NAME = "InvertedPendulum-v5"
# Switch to "Hopper-v5" to try the harder MuJoCo benchmark — see the Traps and
# Fragility section for why Hopper is much harder for an educational-scale
# Dreamer (model exploitation, dense-reward hacking, short random episodes).


def make_env(seed: int = 0):
    try:
        env = gym.make(ENV_NAME)
    except Exception as e:
        print("Failed to create environment:", ENV_NAME)
        print("Error:", e)
        print("\nIf you're missing MuJoCo, try installing:")
        print("  pip install gymnasium[mujoco]")
        raise

    env = RecordEpisodeStatistics(env)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    return env


env = make_env(SEED)
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
act_low = torch.as_tensor(env.action_space.low, device=DEVICE, dtype=torch.float32)
act_high = torch.as_tensor(env.action_space.high, device=DEVICE, dtype=torch.float32)

print("Obs dim:", obs_dim, "Act dim:", act_dim)
print("Action range:", env.action_space.low, "to", env.action_space.high)

## Replay Buffer (sequence sampling)

Unlike the model-free algorithms earlier in this course (DQN, SAC, ...), Dreamer trains the world
model on **contiguous sequences** of fixed length, not on individual transitions $(s, a, r, s')$.
The reason is the **RSSM is recurrent**: it must see a sequence of observations and actions to
unroll its hidden state and compute KL between prior and posterior at every timestep.

This is why we don't reuse `util.rl_algos.ReplayBuffer` here — that buffer stores flat,
independently-sampled transitions (perfect for Bellman-style updates) but cannot return contiguous
chunks from the same episode. So in this notebook we define a small **episode-aware** replay buffer:
it stores full episodes and exposes a `sample_sequences(batch_size, seq_len)` method that returns
batches shaped `[batch, seq_len, ...]`.


In [ ]:
class EpisodeBuffer:
    def __init__(self):
        self.obs = []
        self.acts = []
        self.rews = []
        self.dones = []

    def add(self, obs, act, rew, done):
        self.obs.append(np.asarray(obs, dtype=np.float32))
        self.acts.append(np.asarray(act, dtype=np.float32))
        self.rews.append(np.asarray(rew, dtype=np.float32))
        self.dones.append(np.asarray(done, dtype=np.float32))

    def __len__(self):
        return len(self.obs)


class ReplayBuffer:
    def __init__(self, capacity_episodes: int = 200):
        self.capacity = capacity_episodes
        self.episodes = []
        self.num_steps = 0

    def add_episode(self, ep: EpisodeBuffer):
        if len(ep) < 2:
            return
        self.episodes.append(ep)
        self.num_steps += len(ep)
        if len(self.episodes) > self.capacity:
            removed = self.episodes.pop(0)
            self.num_steps -= len(removed)

    def __len__(self):
        return len(self.episodes)

    def sample_sequences(self, batch_size: int, seq_len: int):
        # Sample random episodes, then random starting indices inside them.
        # TODO: Filter to episodes long enough to yield a full `seq_len` slice.
        #       (If none are eligible we raise below — this is the silent training
        #       stall described in the "Traps and Fragility" section.)
        eligible = None
        if len(eligible) == 0:
            raise ValueError(
                f"No episodes long enough for seq_len={seq_len}. "
                f"Try collecting more data or reducing seq_len."
            )
        # TODO: Sample `batch_size` episodes from `eligible` with replacement.
        #       Hint: `random.choices(population, k=...)`.
        eps = None

        obs = np.zeros((batch_size, seq_len, obs_dim), dtype=np.float32)
        acts = np.zeros((batch_size, seq_len, act_dim), dtype=np.float32)
        rews = np.zeros((batch_size, seq_len, 1), dtype=np.float32)
        dones = np.zeros((batch_size, seq_len, 1), dtype=np.float32)

        for i, ep in enumerate(eps):
            # TODO: Pick a random start index so the slice [start, start+seq_len)
            #       fits inside the episode. Hint: `random.randint` on a valid range.
            start = None

            obs[i] = np.stack(ep.obs[start : start + seq_len], axis=0)
            acts[i] = np.stack(ep.acts[start : start + seq_len], axis=0)
            rews[i, :, 0] = np.stack(ep.rews[start : start + seq_len], axis=0)
            dones[i, :, 0] = np.stack(ep.dones[start : start + seq_len], axis=0)

        # to torch
        obs_t = torch.as_tensor(obs, device=DEVICE)
        acts_t = torch.as_tensor(acts, device=DEVICE)
        rews_t = torch.as_tensor(rews, device=DEVICE)
        dones_t = torch.as_tensor(dones, device=DEVICE)
        return obs_t, acts_t, rews_t, dones_t

## Building blocks

We define a simple MLP factory used throughout the notebook. Every hidden layer includes
**LayerNorm** before the activation — this is what DreamerV3 does, and it is critical for stability:
without it, intermediate activations can grow unboundedly through the network, leading to NaN during
training.

### Symlog and two-hot encoding

DreamerV3 introduces two tools that work together to make the reward head and critic robust:

1. **Symlog/symexp**: $\text{symlog}(x) = \text{sign}(x) \ln(1 + |x|)$ compresses large magnitudes
   while preserving the near-zero region. All prediction targets (observations, rewards, values) are
   mapped into this compressed space before training.

2. **Two-hot encoding**: instead of regressing a single scalar, each prediction head outputs a
   distribution over a fixed grid of 255 bins spanning $[-20, 20]$ in symlog space. The target is a
   "two-hot" vector: probability mass is split between the two bins that bracket the true value,
   weighted by proximity.

Why is this necessary? A scalar MSE head trained on near-constant rewards (like +1 per alive step in
InvertedPendulum) learns to output a constant everywhere — including in out-of-distribution states
the actor pushes into during imagination. The actor then exploits these blind spots via pathwise
gradients and collapses. A distributional head trained with cross-entropy is much harder to exploit:
it learns a sharper, more calibrated mapping, and its gradients are better conditioned.


In [ ]:
def mlp(in_dim: int, hidden: int, out_dim: int, depth: int = 2, act=nn.ELU):
    layers = []
    d = in_dim
    for _ in range(depth):
        layers += [nn.Linear(d, hidden), nn.LayerNorm(hidden), act()]
        d = hidden
    layers += [nn.Linear(d, out_dim)]
    return nn.Sequential(*layers)


def symlog(x: torch.Tensor) -> torch.Tensor:
    """Signed log: compresses large magnitudes without losing the near-zero region."""
    # TODO: Implement symlog(x) = sign(x) * log(1 + |x|).
    #       Hint: `torch.sign`, `torch.log1p`, `torch.abs`.
    return None


def symexp(x: torch.Tensor) -> torch.Tensor:
    """Inverse of symlog."""
    # TODO: Implement symexp(x) = sign(x) * (exp(|x|) - 1).
    return None


# ---- Two-hot distributional encoding (DreamerV3) ----
# A grid of 255 evenly-spaced bins in symlog space. Covers raw values up to
# ~exp(20) ≈ 485 million — far more than any reward or return we'll see.
NUM_BINS = 255
TWOHOT_BINS = torch.linspace(-20, 20, NUM_BINS, device=DEVICE)


def twohot_encode(x: torch.Tensor) -> torch.Tensor:
    """Encode scalars as soft two-hot distributions over TWOHOT_BINS.

    Input:  x of shape [..., 1] (values in symlog space)
    Output: target of shape [..., NUM_BINS] (probability vectors that sum to 1)

    Each target value is placed between the two nearest bins, with probability
    mass split proportionally by distance — like a soft version of one-hot.
    """
    x = x.squeeze(-1)
    x = torch.clamp(x, TWOHOT_BINS[0], TWOHOT_BINS[-1])
    # TODO: Find the lower-bracket bin index for each value.
    #       Hint: `torch.bucketize(x, TWOHOT_BINS)` gives the first bin strictly
    #       greater than x — subtract 1 to get the lower neighbor.
    below = None
    # Clamp so we always have a valid `above` neighbor at `below + 1`.
    below = torch.clamp(below, 0, NUM_BINS - 2)
    above = below + 1
    # TODO: Interpolation weight on the `above` bin: (x - bin_below) / bin_width.
    #       `weight_below` is then 1 - weight_above.
    weight_above = None
    weight_below = 1.0 - weight_above
    # Build target vector
    target = torch.zeros(*x.shape, NUM_BINS, device=x.device)
    target.scatter_(-1, below.unsqueeze(-1), weight_below.unsqueeze(-1))
    target.scatter_(-1, above.unsqueeze(-1), weight_above.unsqueeze(-1))
    return target


def twohot_decode(logits: torch.Tensor) -> torch.Tensor:
    """Decode distribution logits to expected value in symlog space.

    Input:  logits of shape [..., NUM_BINS]
    Output: values of shape [..., 1]

    Simply takes softmax to get probabilities, then the weighted sum over bins.
    """
    # TODO: Softmax `logits` along the last dim to get bin probabilities.
    probs = None
    # TODO: Return the expected value under `probs` over TWOHOT_BINS.
    #       Sum over the last dim and keep it (so output shape is [..., 1]).
    return None

## RSSM — Recurrent State-Space Model

The RSSM is the core of the Dreamer world model. It maintains a latent state with two components:

- A **deterministic** recurrent state $h_t$ (GRU hidden state) — provides long-range memory.
- A **stochastic** latent state $z_t$ (sampled from a learned distribution) — captures uncertainty
  and multimodality in the environment dynamics.

Two distributions are defined over $z_t$:

- **Prior** $p_\theta(z_t \mid h_t)$: what the model predicts _without_ seeing the current
  observation. Used during imagination.
- **Posterior** $q_\theta(z_t \mid h_t, o_t)$: refined with the actual observation. Used during
  training.

The key equations are:

$$h_t = f_\theta(h_{t-1}, z_{t-1}, a_{t-1}) \quad \text{(deterministic GRU transition)}$$
$$p_\theta(z_t \mid h_t) = \mathcal{N}(\mu_p, \sigma_p) \quad \text{(prior)}$$
$$q_\theta(z_t \mid h_t, o_t) = \mathcal{N}(\mu_q, \sigma_q) \quad \text{(posterior)}$$

### Why both deterministic + stochastic?

The deterministic path $h_t$ provides long-range memory (like a standard RNN). The stochastic path
$z_t$ captures uncertainty — the same history can lead to different futures, and the model needs to
represent that distribution. This is analogous to a sequential VAE with a recurrent backbone.

We sample $z_t$ using the **reparameterization trick** ($z = \mu + \sigma \cdot \epsilon$,
$\epsilon \sim \mathcal{N}(0,1)$), which allows gradients to flow through the sampling step into
both the RSSM parameters and (crucially) the actor during imagination.

> **Note on DreamerV2/V3:** later versions replace the Gaussian $z_t$ with **discrete categorical
> latents** (one-hot vectors sampled via the straight-through estimator). This makes the latent
> space harder for the actor to smoothly exploit via pathwise gradients. We use Gaussian latents
> here because they work well at our educational scale with the other anti-exploitation measures in
> place (advantages, target critic, return scaling, two-hot heads). See the Exercises section for
> pointers on implementing categorical latents.


In [ ]:
@dataclass
class RSSMState:
    h: torch.Tensor  # deterministic state [B, deter_dim]
    z: torch.Tensor  # stochastic state     [B, stoch_dim]


class RSSM(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        act_dim: int,
        deter_dim: int = 128,
        stoch_dim: int = 32,
        hidden: int = 128,
    ):
        super().__init__()
        self.deter_dim = deter_dim
        self.stoch_dim = stoch_dim

        # GRU updates deterministic state given previous (z, action)
        self.gru = nn.GRUCell(stoch_dim + act_dim, deter_dim)
        # LayerNorm on GRU output prevents hidden state explosion during long
        # sequences. This is a key DreamerV3 stabilization technique.
        self.gru_norm = nn.LayerNorm(deter_dim)

        # TODO: Build the prior MLP p(z | h). Input: deter_dim. Output:
        #       2 * stoch_dim values (mean and raw_std for the Gaussian).
        self.prior_net = None

        # TODO: Build the posterior MLP q(z | h, embed(obs)). Input:
        #       deter_dim + embed_dim. Output: 2 * stoch_dim.
        self.post_net = None

    def init_state(self, batch_size: int) -> RSSMState:
        h = torch.zeros(batch_size, self.deter_dim, device=DEVICE)
        z = torch.zeros(batch_size, self.stoch_dim, device=DEVICE)
        return RSSMState(h=h, z=z)

    def _dist_from_stats(self, stats: torch.Tensor) -> Normal:
        # TODO: Split `stats` into (mean, raw_std) halves along the last dim.
        #       Hint: `torch.chunk(..., chunks=2, dim=-1)`.
        mean, std = None, None
        # TODO: Apply softplus to `std` and add a small floor (e.g. 0.01) to
        #       keep it strictly positive and avoid degenerate distributions.
        std = None
        return Normal(mean, std)

    def prior(self, h: torch.Tensor) -> Normal:
        # TODO: Pass h through `prior_net` and wrap with `_dist_from_stats`.
        return None

    def posterior(self, h: torch.Tensor, embed: torch.Tensor) -> Normal:
        # TODO: Concatenate h and embed along the last dim, run through
        #       `post_net`, and wrap with `_dist_from_stats`.
        return None

    def img_step(self, prev: RSSMState, action: torch.Tensor) -> Tuple[RSSMState, Normal]:
        """One imagination step: update h with GRU, sample z from the PRIOR."""
        # TODO: Build the GRU input by concatenating prev.z and action.
        x = None
        # TODO: Step the GRU (input x, previous hidden prev.h), then apply
        #       `gru_norm` for stability.
        h = None
        # TODO: Compute the prior distribution over z given h, and rsample()
        #       (NOT sample) so gradients flow through the actor pathway.
        prior_dist = None
        z = None
        return RSSMState(h=h, z=z), prior_dist

    def obs_step(
        self, prev: RSSMState, action: torch.Tensor, embed: torch.Tensor
    ) -> Tuple[RSSMState, Normal, Normal]:
        """One observation step: update h, compute prior + posterior, sample z from POSTERIOR."""
        # TODO: Same GRU+LayerNorm update as `img_step`.
        x = None
        h = None
        # TODO: Compute both the prior (from h alone) and the posterior
        #       (from h + embed). We need both for the KL loss in training.
        prior_dist = None
        post_dist = None
        # TODO: rsample z from the POSTERIOR (the observation-refined one).
        z = None
        return RSSMState(h=h, z=z), post_dist, prior_dist

    def features(self, state: RSSMState) -> torch.Tensor:
        # TODO: Concatenate h and z along the last dim — these are the "latent
        #       features" fed to every prediction head, the actor, and the critic.
        return None

## World Model

The world model wraps the RSSM with an encoder and three prediction heads. Given latent features
$\text{feat}_t = [h_t, z_t]$, we predict:

- **Observation reconstruction** $\hat{o}_t$: ensures the latent state captures useful information
  about the real observation (trained with MSE on symlog-compressed targets).
- **Reward** $\hat{r}_t$: needed for computing returns during imagination. Uses a **two-hot
  distributional head** — the network outputs logits over 255 symlog-space bins, trained with
  cross-entropy against the two-hot encoding of symlog(reward).
- **Continuation** $\hat{c}_t \in (0,1)$: the probability that the episode continues. This replaces
  the usual fixed discount $\gamma$ with a learned, state-dependent discount
  $\gamma \cdot \hat{c}_t$, which is important for tasks with variable-length episodes (like Hopper,
  where the robot can fall).

The full world model loss is:

$$
\mathcal{L}_{WM} = \underbrace{\|o_t - \hat{o}_t\|^2}_{\text{reconstruction}}
  + \underbrace{\text{CE}_{\text{two-hot}}(r_t, \hat{r}_t)}_{\text{reward}}
  + \underbrace{\text{BCE}(c_t, \hat{c}_t)}_{\text{continuation}}
  + \underbrace{\beta \cdot \mathcal{L}_{KL}}_{\text{KL regularization}}
$$

### KL balancing (DreamerV2)

A naive KL term $\text{KL}[q(z|h,o) \| p(z|h)]$ can be unstable: as the posterior becomes very
confident (low variance), the KL shoots up exponentially, causing gradient explosions.

DreamerV2 introduced **KL balancing** to fix this. The KL loss is split into two parts:

$$\mathcal{L}_{KL} = \alpha \cdot \text{KL}[\text{sg}(q) \| p] + (1-\alpha) \cdot \text{KL}[q \| \text{sg}(p)]$$

where $\text{sg}$ is stop-gradient and $\alpha = 0.8$. The first term trains the **prior** to match
the posterior (harmless — makes imagination more accurate). The second term **lightly regularizes**
the posterior (prevents it from collapsing). Since only 20% of the gradient pushes the posterior, it
can't be driven to an extreme that causes KL explosion.

We also apply a **free-nats** threshold: KL values below `free_nats` are not penalized. This
prevents posterior collapse early in training, where the model would ignore the stochastic state
entirely.


In [ ]:
class WorldModel(nn.Module):
    def __init__(
        self,
        obs_dim: int,
        act_dim: int,
        embed_dim: int = 64,
        deter_dim: int = 128,
        stoch_dim: int = 32,
        hidden: int = 128,
    ):
        super().__init__()
        # TODO: Build the observation encoder (mlp) from obs_dim -> embed_dim.
        self.encoder = None
        self.rssm = RSSM(
            embed_dim=embed_dim,
            act_dim=act_dim,
            deter_dim=deter_dim,
            stoch_dim=stoch_dim,
            hidden=hidden,
        )

        feat_dim = deter_dim + stoch_dim
        # TODO: Build the observation decoder (mlp, depth=3) from feat_dim -> obs_dim.
        self.decoder = None
        # TODO: Build the reward head. KEY DreamerV3 detail: it outputs NUM_BINS
        #       logits (for the two-hot distributional target), NOT a scalar. Use depth=3.
        self.reward_head = None
        # TODO: Build the continuation head (mlp, depth=3) from feat_dim -> 1.
        #       Outputs raw logits; sigmoid is applied when we want a probability.
        self.cont_head = None

    def forward_posterior(self, obs_seq: torch.Tensor, act_seq: torch.Tensor):
        """Run RSSM through a sequence using the posterior (uses observations).

        obs_seq: [B, T, obs_dim]
        act_seq: [B, T, act_dim]
        Returns dict of tensors with time dimension.
        """
        B, T, _ = obs_seq.shape
        # TODO: Encode every observation. Flatten batch+time into one dim,
        #       run through `self.encoder`, then reshape back to [B, T, embed_dim].
        embeds = None

        # Dreamer-style indexing: use previous action a_{t-1} to update state at time t.
        # TODO: Build `act_prev` by prepending a zero action and dropping the last one
        #       along the time dim. Resulting shape: [B, T, act_dim].
        zero_act = torch.zeros(B, 1, act_seq.shape[-1], device=act_seq.device, dtype=act_seq.dtype)
        act_prev = None

        state = self.rssm.init_state(B)
        states = []
        post_dists = []
        prior_dists = []

        for t in range(T):
            # TODO: Advance the RSSM by one posterior step using `act_prev[:, t]`
            #       and `embeds[:, t]`. Append the new state and both distributions
            #       (post, prior) to the lists above.
            pass

        # stack features
        h = torch.stack([s.h for s in states], dim=1)
        z = torch.stack([s.z for s in states], dim=1)
        feat = torch.cat([h, z], dim=-1)  # [B, T, feat_dim]

        return {
            "h": h,
            "z": z,
            "feat": feat,
            "post_dists": post_dists,
            "prior_dists": prior_dists,
        }

    def decode(self, feat: torch.Tensor) -> torch.Tensor:
        return self.decoder(feat)

    def predict_reward(self, feat: torch.Tensor) -> torch.Tensor:
        """Reward logits over NUM_BINS symlog-space bins."""
        return self.reward_head(feat)

    def predict_cont_logits(self, feat: torch.Tensor) -> torch.Tensor:
        """Raw logits for continuation probability (use with BCE-with-logits for stability)."""
        return self.cont_head(feat)

    def predict_cont(self, feat: torch.Tensor) -> torch.Tensor:
        """Continuation probability in (0, 1)."""
        return torch.sigmoid(self.cont_head(feat))

## Actor & Critic (in latent space)

Unlike model-free methods where the actor and critic take _observations_ as input, Dreamer's actor
and critic operate on **latent features** $[h_t, z_t]$. They never see raw observations.

We use a **tanh-squashed Gaussian** policy (common in continuous control). The actor maps latent
features to a distribution over actions:

$$\pi_\phi(a_t \mid h_t, z_t) = \text{TanhNormal}\big(\mu_\phi(\text{feat}_t),\; \sigma_\phi(\text{feat}_t)\big)$$

The actor's objective is to maximize imagined returns plus an entropy bonus:

$$\max_\phi \; \mathbb{E}_{\text{imagine}}\Big[\sum_t \gamma^t \big(R_t^\lambda - V(s_t) + \eta\,\mathcal{H}[\pi]\big)\Big]$$

The critical Dreamer insight: **gradients flow through the world model dynamics into the actor**.
When the actor changes its action, the imagined next state changes (via the RSSM), which changes the
predicted reward, which changes the return. This is fundamentally different from model-free policy
gradients, which use REINFORCE-style score function estimators.

The **critic** learns a value function $V_\psi(\text{feat}_t)$ by regression to TD($\lambda$)
targets computed over the imagined trajectories.


In [ ]:
class TanhGaussianPolicy(nn.Module):
    def __init__(self, feat_dim: int, act_dim: int, hidden: int = 256, max_std: float = 1.0):
        super().__init__()
        # Network outputs mean and raw_std (before sigmoid transform).
        self.net = mlp(feat_dim, hidden, 2 * act_dim, depth=3)
        self.act_dim = act_dim
        self.min_std = 0.1
        self.max_std = max_std

    def dist(self, feat: torch.Tensor) -> Normal:
        stats = self.net(feat)
        # TODO: Split `stats` into (mean, raw_std) along the last dim.
        mean, raw_std = None, None
        # DreamerV3 sigmoid-bounded std: sigmoid has nonzero gradient everywhere,
        # so the entropy bonus can always push std back up. A hard clamp would
        # create a gradient dead zone where std gets stuck at its minimum.
        # TODO: Map raw_std into [min_std, max_std] via sigmoid:
        #       std = min_std + (max_std - min_std) * sigmoid(raw_std).
        std = None
        return Normal(mean, std)

    def sample(self, feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample a tanh-squashed action and return its analytical entropy.

        We return the **analytical entropy of the base Normal** rather than a
        single-sample MC estimate `-log p(a)`. The Monte-Carlo estimate is very
        noisy, especially when the action saturates near ±1 (the tanh
        change-of-variables term blows up), and using it as an entropy bonus
        gives misleading regularization. The base-Normal entropy is what we
        actually want to keep high: the spread of the *pre-squash* policy.
        """
        base_dist = self.dist(feat)
        # TODO: rsample() from `base_dist` so gradients flow through the sample
        #       (pathwise gradient — essential for Dreamer's actor update).
        a = None
        # TODO: Apply tanh to squash the action into [-1, 1].
        a_tanh = None
        # TODO: Analytical entropy of the BASE Normal (not the squashed one).
        #       Sum across action dims, keep the last dim so shape is [..., 1].
        entropy = None
        return a_tanh, entropy

    def mode(self, feat: torch.Tensor) -> torch.Tensor:
        # TODO: Deterministic action: tanh of the distribution mean.
        return None


class ValueNet(nn.Module):
    """Two-hot distributional critic: outputs logits over NUM_BINS symlog-space bins."""

    def __init__(self, feat_dim: int, hidden: int = 256):
        super().__init__()
        self.net = mlp(feat_dim, hidden, NUM_BINS, depth=3)

    def forward(self, feat: torch.Tensor) -> torch.Tensor:
        return self.net(feat)

## Hyperparameters

These defaults are tuned for InvertedPendulum-v5. The Hopper training cell below overrides several
of them (larger model, longer horizon, adjusted learning rates and loss scales). Key parameters to
understand:

- `train_steps`: total environment interactions. Increase for harder tasks.
- `horizon`: imagination rollout length. Longer = more signal but more compounding error.
- `train_ratio`: gradient steps per environment step. Higher = more sample-efficient but slower.
- `reward_scale`: how much weight the WM places on reward prediction vs. reconstruction. With
  high-dimensional observations, the WM can neglect reward accuracy unless this is increased.
- `kl_scale` / `free_nats`: control how tightly the prior tracks the posterior.
- `gamma`: discount factor. Higher values (e.g. 0.997) help on tasks where payoff is spread over
  many future steps (locomotion).


In [ ]:
@dataclass
class Config:
    # data collection
    seed_steps: int = 2500  # random exploration steps before the actor starts
    train_steps: int = 50_000  # total real environment steps
    wm_pretrain_steps: int = 500  # WM-only gradient updates between seed and
    # main training. Without this, the actor's very first gradient flows
    # through a near-random WM and optimizes noise.

    # model sizes
    deter_dim: int = 128  # RSSM deterministic state dimension
    stoch_dim: int = 32  # RSSM stochastic state dimension (Gaussian)
    hidden_dim: int = 128  # MLP hidden layer width (WM, encoder, decoder)
    max_std: float = 1.0  # upper bound on actor std (sigmoid parameterization)

    # replay / sequences
    capacity_episodes: int = 3000  # keep all episodes for a typical run
    batch_size: int = 32
    seq_len: int = 32  # InvertedPendulum episodes are 200 steps — plenty of room

    # model-based imagination
    horizon: int = 5  # default for IP; Hopper overrides to 15 (the paper's value).
    # Shorter horizons compound less WM error; the TD(lambda) bootstrap
    # still picks up long-term value via the critic.
    gamma: float = 0.99
    lam: float = 0.95

    # training ratio: how many gradient updates per environment step
    train_ratio: int = 1

    # actor throttling: update the actor/critic once every N world-model
    # updates. With a small WM, the actor can exploit model errors faster
    # than the WM can correct them. Giving the WM a head start keeps the
    # imagined returns grounded in reality.
    actor_update_every: int = 4

    # learning rates — DreamerV3 paper defaults. Lower than our earlier values;
    # prevents the policy/WM coevolution instability where the actor finds a
    # good policy and then loses it as the WM drifts under it.
    lr_wm: float = 3e-4
    lr_actor: float = 3e-5
    lr_value: float = 3e-5

    # loss weights
    kl_scale: float = 1.0
    free_nats: float = 1.0
    kl_balance: float = 0.8  # DreamerV2: 80% trains prior, 20% regularizes posterior
    reward_scale: float = 1.0
    cont_scale: float = 1.0
    recon_scale: float = 1.0

    # actor entropy bonus (scale-invariant: returns are rescaled by EMA of
    # P95-P5 spread before use). DreamerV3 uses 3e-4 with large world models;
    # our small WM (128/128) is easier to exploit, so we use a higher scale.
    # The sigmoid std parameterization prevents hard entropy collapse even at
    # this moderate scale.
    entropy_scale: float = 1e-2
    return_scale_decay: float = 0.99  # EMA decay for the return-scale tracker

    # optimization
    grad_clip: float = 10.0
    wm_weight_decay: float = 1e-5  # WD on WM fights memorization of replay

    # target critic (Polyak EMA) — stabilizes the TD(lambda) bootstrap
    target_tau: float = 0.02

    # logging / eval. We use 15 eval episodes because Pendulum's random initial
    # states have very high variance: 5 episodes gives unreliable eval means
    # where a single bad start can drag the reported mean down by ~500.
    eval_every_steps: int = 2000
    eval_episodes: int = 15


CFG = Config()
CFG

## Training utilities

Helper functions for:

- **Action mapping**: converting tanh-space actions $[-1, 1]$ to the environment's action range.
- **Evaluation**: running the learned policy in a fresh environment (using the posterior to track
  state, and the actor's _mode_ for deterministic actions).
- **TD($\lambda$) returns**: the recursive target computation used in imagined rollouts.


In [ ]:
def to_action_env(a_tanh: torch.Tensor) -> np.ndarray:
    """Map tanh action in [-1,1] to env action space (Box)."""
    # scale to [low, high]
    a = act_low + (a_tanh + 1.0) * 0.5 * (act_high - act_low)
    return a.detach().cpu().numpy()


@torch.no_grad()
def eval_policy(
    env_name: str,
    wm: WorldModel,
    actor: TanhGaussianPolicy,
    episodes: int = 5,
    render: bool = False,
):
    # Create a fresh env for evaluation (separate from the training env).
    try:
        e = gym.make(env_name)
    except Exception as err:
        print("Failed to create evaluation environment:", env_name)
        print("Error:", err)
        raise
    e = RecordEpisodeStatistics(e)
    e.reset(seed=SEED + 123)
    e.action_space.seed(SEED + 123)
    returns = []
    for _ in range(episodes):
        obs, _ = e.reset()
        done = False
        ep_ret = 0.0

        # We need an RSSM state. We'll run posterior with a dummy action at the first step.
        state = wm.rssm.init_state(batch_size=1)
        prev_action = torch.zeros(1, act_dim, device=DEVICE)

        while not done:
            obs_t = torch.as_tensor(obs, device=DEVICE).float().unsqueeze(0)
            embed = wm.encoder(obs_t)
            state, post, prior = wm.rssm.obs_step(state, prev_action, embed)

            feat = wm.rssm.features(state)
            a_tanh = actor.mode(feat)
            action = to_action_env(a_tanh)[0]

            obs, rew, terminated, truncated, info = e.step(action)
            done = terminated or truncated
            ep_ret += float(rew)
            prev_action = torch.as_tensor(action, device=DEVICE).float().unsqueeze(0)

            if render:
                e.render()

        returns.append(ep_ret)
    e.close()
    return float(np.mean(returns)), float(np.std(returns))


def lambda_return(
    rewards: torch.Tensor,
    values: torch.Tensor,
    discounts: torch.Tensor,
    bootstrap: torch.Tensor,
    lam: float,
) -> torch.Tensor:
    """TD(lambda) returns.

    rewards:   [B, H, 1]
    values:    [B, H, 1]   (V(s_t))
    discounts: [B, H, 1]
    bootstrap: [B, 1]      (V(s_H))
    returns:   [B, H, 1]
    """
    # Following Dreamer-style recursion from the end.
    # TODO: Build `next_values` = V(s_{t+1}) by shifting `values` left by one step
    #       and appending `bootstrap` at the tail. Final shape [B, H, 1].
    #       Hint: torch.cat([values[:, 1:], bootstrap.unsqueeze(1)], dim=1).
    next_values = None
    # TODO: Per-step "target" = r_t + discount_t * (1 - lam) * V(s_{t+1}).
    #       This is the non-bootstrapped part of the lambda-return recursion.
    targets = None

    # TODO: Accumulate the lambda-return from the end. Starting from `ret = bootstrap`:
    #         ret_{t} = targets[:, t] + discount_t * lam * ret_{t+1}
    #       Append each `ret` to `rets`; we'll reverse at the end.
    ret = bootstrap
    rets = []
    for t in reversed(range(rewards.shape[1])):
        # TODO: Update `ret` and append to `rets`.
        pass
    rets = torch.stack(list(reversed(rets)), dim=1)
    return rets

## Instantiate models & optimizers

We create separate optimizers for the world model, actor, and critic. This is standard in Dreamer:
the world model is trained on real data, while the actor and critic are trained on imagined data —
keeping their gradients separate avoids interference.


In [ ]:
wm = WorldModel(obs_dim=obs_dim, act_dim=act_dim).to(DEVICE)
feat_dim = wm.rssm.deter_dim + wm.rssm.stoch_dim

actor = TanhGaussianPolicy(feat_dim=feat_dim, act_dim=act_dim).to(DEVICE)
value_net = ValueNet(feat_dim=feat_dim).to(DEVICE)

# Target critic for stable TD(lambda) bootstrapping. The actor/critic update
# uses this slow-moving copy when computing bootstrapped returns, and we Polyak-
# average it toward the live critic after each gradient step. Without this the
# critic's own current output feeds back into its own regression target, which
# causes the unstable oscillations you see in naive Dreamer implementations.
value_net_target = ValueNet(feat_dim=feat_dim).to(DEVICE)
value_net_target.load_state_dict(value_net.state_dict())
for p in value_net_target.parameters():
    p.requires_grad = False

opt_wm = torch.optim.AdamW(wm.parameters(), lr=CFG.lr_wm, weight_decay=CFG.wm_weight_decay)
opt_actor = torch.optim.Adam(actor.parameters(), lr=CFG.lr_actor)
opt_value = torch.optim.Adam(value_net.parameters(), lr=CFG.lr_value)

replay = ReplayBuffer(capacity_episodes=CFG.capacity_episodes)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


print(f"World model: {count_params(wm):,} params")
print(f"Actor:       {count_params(actor):,} params")
print(f"Critic:      {count_params(value_net):,} params")
print(f"Total:       {count_params(wm) + count_params(actor) + count_params(value_net):,} params")

## World model update

We train the world model on sequences sampled from the replay buffer. For each sequence, the RSSM
runs forward using the posterior (which sees observations), and we minimize four losses:

- **Reconstruction**: MSE between predicted and actual observations (in symlog space).
- **Reward**: cross-entropy between the two-hot distributional head's logits and the two-hot
  encoding of symlog(reward). This is more robust than scalar MSE — a distributional head can't
  simply "memorize the mean" and must learn a calibrated prediction for each state.
- **Continuation**: binary cross-entropy between predicted and actual episode-continuation signals
  (using logits for numerical stability).
- **KL with balancing**: the balanced KL loss described above, with free-nats threshold.

A note on indexing: the RSSM state at time $t$ is computed using the _previous_ action $a_{t-1}$, so
reward predictions at time $t$ correspond to $r_{t-1}$ from the replay buffer.


In [ ]:
def world_model_update(obs_seq, act_seq, rew_seq, done_seq) -> Dict[str, float]:
    out = wm.forward_posterior(obs_seq, act_seq)
    feat = out["feat"]  # [B, T, feat_dim]

    # Prediction heads
    obs_pred = wm.decode(feat)  # [B, T, obs_dim]
    rew_logits = wm.predict_reward(feat)  # [B, T, NUM_BINS]
    cont_logits = wm.predict_cont_logits(feat)  # [B, T, 1]

    # TODO: Reconstruction loss = MSE between `obs_pred` and `symlog(obs_seq)`.
    #       Dreamer decodes into symlog space, not raw obs space — this compresses
    #       outliers so the loss isn't dominated by a few huge states.
    recon_loss = None

    B, T, _ = rew_seq.shape
    if T > 1:
        # The reward head predicts r_{t+1} from feat_t, so `rew_logits[:, 1:]`
        # corresponds to `rew_seq[:, :-1]` (reward indexed against the previous action).
        # TODO: Two-hot target from symlog(rew_seq[:, :-1]). Shape [B, T-1, NUM_BINS].
        rew_target = None
        # TODO: log_softmax of `rew_logits[:, 1:]` along the last dim.
        rew_log_probs = None
        # TODO: Cross-entropy: negative sum over bins of (target * log_probs), then .mean().
        reward_loss = None

        # TODO: Continuation prediction follows the same transition convention as the reward
        #       head: `cont_logits[:, 1:]` (read from feature s_{t+1}) predicts whether the
        #       episode kept going past s_{t+1}, i.e. `1 - done_seq[:, :-1]`. Build the
        #       shifted target tensor here.
        cont_target = None
        # TODO: BCE-with-logits between `cont_logits[:, 1:]` and `cont_target` (numerically
        #       stable — don't sigmoid then BCE).
        cont_loss = None
    else:
        reward_loss = torch.zeros((), device=DEVICE)
        cont_loss = torch.zeros((), device=DEVICE)

    # --- KL balancing (DreamerV2) ---
    # Split the KL gradient 80% on the prior, 20% on the posterior. Stop-gradient
    # on the opposite side in each half so we push only one distribution at a time.
    kl_for_prior = []
    kl_for_post = []
    kl_raw = []
    for post, prior in zip(out["post_dists"], out["prior_dists"]):
        # TODO: `post_sg` — same posterior distribution but with mean and stddev
        #       .detach()'d (stop-gradient). Build a new Normal from those.
        post_sg = None
        # TODO: `prior_sg` — same for the prior.
        prior_sg = None
        # TODO: KL(post_sg || prior) — trains ONLY the prior. Sum over stoch dims
        #       with keepdim=True, so shape is [B, 1].
        kl_for_prior.append(None)
        # TODO: KL(post || prior_sg) — lightly regularizes ONLY the posterior.
        kl_for_post.append(None)
        kl_raw.append(torch.distributions.kl.kl_divergence(post, prior).sum(dim=-1, keepdim=True))
    kl_prior = torch.stack(kl_for_prior, dim=1)
    kl_post = torch.stack(kl_for_post, dim=1)
    kl_raw_t = torch.stack(kl_raw, dim=1)

    # TODO: Balanced KL loss with free-nats floor:
    #         CFG.kl_balance       * clamp(kl_prior - CFG.free_nats, min=0).mean()
    #       + (1 - CFG.kl_balance) * clamp(kl_post  - CFG.free_nats, min=0).mean()
    kl_loss = None

    # TODO: Total world-model loss: sum of the four pieces, each scaled by its
    #       CFG weight (recon_scale, reward_scale, cont_scale, kl_scale).
    loss = None

    opt_wm.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(wm.parameters(), CFG.grad_clip)
    if torch.isfinite(grad_norm):
        opt_wm.step()

    return {
        "wm/loss": float(loss.item()),
        "wm/recon": float(recon_loss.item()),
        "wm/reward": float(reward_loss.item()),
        "wm/cont": float(cont_loss.item()),
        "wm/kl_raw": float(kl_raw_t.mean().item()),
        "wm/kl_loss": float(kl_loss.item()),
    }

## Imagination + Actor/Critic update

This is where Dreamer's model-based advantage shines. Starting from posterior states obtained from
real sequences, we:

1. **Imagine** forward for `horizon` steps using only the **prior** (no observations needed).
2. **Predict** rewards and continuation probabilities at each imagined step. Rewards are decoded
   from the two-hot distributional head (softmax → expected value → symexp back to real scale).
3. **Compute TD($\lambda$) returns** over the imagined trajectory.
4. **Update the critic** by cross-entropy against two-hot targets of symlog(returns).
5. **Update the actor** to maximize scaled returns plus an entropy bonus.

The TD($\lambda$) return blends 1-step and Monte Carlo targets:

$$R_t^\lambda = r_t + \gamma c_t \Big[(1-\lambda)\, V(s_{t+1}) + \lambda\, R_{t+1}^\lambda\Big]$$

where $c_t$ is the predicted continuation and $\gamma$ is the base discount. Higher $\lambda$ leans
toward Monte Carlo (lower bias, higher variance); lower $\lambda$ leans toward bootstrapping (higher
bias, lower variance).

Each imagined step is weighted by $w_t = \prod_{i<t} \gamma c_i$, which down-weights states that are
far into the future or likely past a terminal state.


In [ ]:
def to_action_tensor(a_tanh: torch.Tensor) -> torch.Tensor:
    """Differentiable mapping from tanh action in [-1,1] to env action space."""
    return act_low + (a_tanh + 1.0) * 0.5 * (act_high - act_low)


def imagine_rollout(start_state: RSSMState, horizon: int):
    """Roll out in latent space using the policy and RSSM prior.

    Returns:
      feats:     features of *current* states s_t,    [B, H, feat_dim]
      actions:   tanh-space actions,                  [B, H, act_dim]
      entropies: analytical Normal entropy of pi(.|s_t), [B, H, 1]
      rewards:   predicted r_{t+1} (real-reward scale), [B, H, 1]
      conts:     predicted continue at s_{t+1},       [B, H, 1]
      last_feat: features of s_H,                     [B, feat_dim]
    """
    B = start_state.h.shape[0]
    state = RSSMState(h=start_state.h, z=start_state.z)

    feats, actions, entropies, rewards, conts = [], [], [], [], []

    for _ in range(horizon):
        # TODO: Latent features of the current state.
        feat = None
        # TODO: Sample an action from the actor using `feat`. `sample` returns
        #       both the tanh-squashed action and its analytical base-Normal entropy.
        a_tanh, entropy = None, None
        # TODO: Map `a_tanh` into the env action range. Use the *differentiable*
        #       `to_action_tensor` (NOT `to_action_env`) so gradients flow.
        a_env = None

        # TODO: Step the RSSM forward in imagination (PRIOR only, no observation)
        #       using `img_step` on the current state and `a_env`.
        next_state, _prior = None, None
        # TODO: Features of the imagined next state s_{t+1}.
        next_feat = None

        # TODO: Predict r_{t+1} from next_feat. The reward head outputs two-hot
        #       logits in SYMLOG space; decode to an expected symlog value, then
        #       symexp back to real-reward scale.
        r = None
        # TODO: Predict the continuation probability at s_{t+1} using
        #       `predict_cont` (already applies sigmoid).
        c = None

        feats.append(feat)
        actions.append(a_tanh)
        entropies.append(entropy)
        rewards.append(r)
        conts.append(c)

        state = next_state

    feats = torch.stack(feats, dim=1)  # [B, H, feat_dim]
    actions = torch.stack(actions, dim=1)  # [B, H, act_dim]
    entropies = torch.stack(entropies, dim=1)  # [B, H, 1]
    rewards = torch.stack(rewards, dim=1)  # [B, H, 1]
    conts = torch.stack(conts, dim=1)  # [B, H, 1]
    last_feat = wm.rssm.features(state)  # [B, feat_dim]
    return feats, actions, entropies, rewards, conts, last_feat


# Module-level state for DreamerV3-style return scaling.
_return_scale = {"ema": 1.0}


def actor_critic_update(obs_seq, act_seq, rew_seq, done_seq) -> Dict[str, float]:
    # Starting latent states from real data (no gradients for this part).
    with torch.no_grad():
        out = wm.forward_posterior(obs_seq, act_seq)
        h = out["h"]  # [B, T, deter_dim]
        z = out["z"]  # [B, T, stoch_dim]

        # Start imagination from random time indices inside the sequence.
        B, T, _ = h.shape
        idx = torch.randint(low=0, high=T, size=(B,), device=DEVICE)
        h0 = h[torch.arange(B, device=DEVICE), idx]
        z0 = z[torch.arange(B, device=DEVICE), idx]
        start = RSSMState(h=h0, z=z0)

    feats, actions, entropies, rewards, conts, last_feat = imagine_rollout(start, CFG.horizon)

    # TODO: Per-step discounts = gamma * predicted continuation. Gradients flow
    #       through `conts` into the actor via pathwise backprop.
    discounts = None  # [B, H, 1]

    # TODO: Trajectory weights w_t = prod_{i<t} discount_i. Hint: start the
    #       sequence with ones, then cumprod. Build it from
    #       `torch.cat([ones(B, 1, 1), discounts[:, :-1]], dim=1)` and `cumprod(dim=1)`.
    #       Detach — these weights are just for loss aggregation and shouldn't
    #       carry their own gradient.
    weights = None

    # Bootstrap values from TARGET critic. Decode two-hot logits to expected
    # symlog value, then symexp to real scale.
    with torch.no_grad():
        # TODO: values_target = symexp(twohot_decode(value_net_target(feats))).  [B, H, 1]
        values_target = None
        # TODO: bootstrap = symexp(twohot_decode(value_net_target(last_feat))).unsqueeze(1).  [B, 1, 1]
        bootstrap = None

    # TODO: TD(lambda) returns over the imagined trajectory using `lambda_return`.
    #       Pass `bootstrap.squeeze(1)` so it has shape [B, 1].
    returns = None  # [B, H, 1]

    # ---- critic update: two-hot cross-entropy on symlog(returns) ----
    # TODO: Live critic logits at the (detached) imagined features. [B, H, NUM_BINS].
    value_logits = None
    # TODO: Two-hot target from symlog(returns.detach()). [B, H, NUM_BINS].
    value_target = None
    # TODO: log_softmax of `value_logits` along the last dim.
    value_log_probs = None
    # TODO: Weighted cross-entropy loss. Scalar.
    #       Hint: (weights.squeeze(-1) * -(value_target * value_log_probs).sum(-1)).mean()
    value_loss = None

    opt_value.zero_grad(set_to_none=True)
    value_loss.backward()
    grad_norm_v = torch.nn.utils.clip_grad_norm_(value_net.parameters(), CFG.grad_clip)
    if torch.isfinite(grad_norm_v):
        opt_value.step()

    # ---- DreamerV3 return scaling for the actor only ----
    with torch.no_grad():
        flat = returns.flatten()
        p95 = torch.quantile(flat, 0.95).item()
        p5 = torch.quantile(flat, 0.05).item()
        spread = max(1.0, p95 - p5)
        _return_scale["ema"] = (
            CFG.return_scale_decay * _return_scale["ema"] + (1.0 - CFG.return_scale_decay) * spread
        )
    # DreamerV3 uses advantages (returns - values), not raw returns. This is
    # critical: without the value baseline, always-positive returns push the
    # actor to exploit every WM error. With advantages, when the WM inflates
    # imagined returns the critic inflates too, so advantages stay centered
    # near zero and the actor only responds to genuine improvements.
    # TODO: advantages = (returns - values_target) / max(1.0, _return_scale["ema"]).
    advantages = None

    # ---- actor update (dynamics backprop) ----
    # TODO: Actor loss = - mean of weights * (advantages + CFG.entropy_scale * entropies).
    #       Negative because we MAXIMIZE scaled advantages + entropy bonus.
    actor_loss = None

    opt_actor.zero_grad(set_to_none=True)
    actor_loss.backward()
    grad_norm_a = torch.nn.utils.clip_grad_norm_(actor.parameters(), CFG.grad_clip)
    if torch.isfinite(grad_norm_a):
        opt_actor.step()
    # Zero WM grads that leaked from pathwise actor backward.
    wm.zero_grad(set_to_none=True)

    # ---- Polyak EMA update of the target critic ----
    with torch.no_grad():
        tau = CFG.target_tau
        for p_live, p_tgt in zip(value_net.parameters(), value_net_target.parameters()):
            p_tgt.data.mul_(1.0 - tau).add_(p_live.data, alpha=tau)

    # ---- Diagnostics ----
    with torch.no_grad():
        actor_std = actor.dist(feats.detach()).stddev.mean().item()
        adv_mean = advantages.mean().item()
        adv_std = advantages.std().item()
        ent_term = (CFG.entropy_scale * entropies).mean().item()

    return {
        "ac/actor_loss": float(actor_loss.item()),
        "ac/value_loss": float(value_loss.item()),
        "ac/return_mean": float(returns.mean().item()),
        "ac/return_scale": float(_return_scale["ema"]),
        "ac/entropy": float(entropies.mean().item()),
        "ac/cont_mean": float(conts.mean().item()),
        "ac/reward_pred_mean": float(rewards.mean().item()),
        "ac/actor_std": actor_std,
        "ac/adv_mean": adv_mean,
        "ac/adv_std": adv_std,
        "ac/ent_term": ent_term,
    }

## Sanity checks (shapes & gradient flow)

Before running the full training loop, we verify that everything is wired up correctly using
**synthetic data** (no environment interaction needed):

1. **Shapes**: the world model's outputs match expected dimensions.
2. **Losses**: all loss terms are finite.
3. **Gradient flow**: gradients reach the actor _through_ the imagined rollout. This is the key
   Dreamer property — if the actor's gradient norm is zero, something is broken in the imagination
   pipeline.


In [ ]:
# --- Sanity checks: shapes & gradient flow (no env interaction needed) ---
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

B, T = 4, 8
obs_seq = torch.randn(B, T, obs_dim, device=DEVICE)
u = torch.rand(B, T, act_dim, device=DEVICE)
act_seq = act_low + u * (act_high - act_low)
rew_seq = torch.randn(B, T, 1, device=DEVICE)
done_seq = torch.zeros(B, T, 1, device=DEVICE)

# 1) World model forward shapes
out = wm.forward_posterior(obs_seq, act_seq)
assert out["feat"].shape[:2] == (B, T)
assert out["feat"].shape[-1] == wm.rssm.deter_dim + wm.rssm.stoch_dim
assert len(out["post_dists"]) == T and len(out["prior_dists"]) == T

obs_pred = wm.decode(out["feat"])
rew_logits = wm.predict_reward(out["feat"])
cont_logits = wm.predict_cont_logits(out["feat"])

assert obs_pred.shape == obs_seq.shape
assert rew_logits.shape == (
    B,
    T,
    NUM_BINS,
), f"Expected reward logits shape {(B, T, NUM_BINS)}, got {rew_logits.shape}"
assert cont_logits.shape == (B, T, 1)

# 2) Two-hot encode/decode round-trip
test_vals = torch.tensor([[0.5], [-3.0], [10.0]], device=DEVICE)
encoded = twohot_encode(test_vals)
assert encoded.shape == (3, NUM_BINS)
assert torch.allclose(encoded.sum(-1), torch.ones(3, device=DEVICE), atol=1e-5)
decoded = twohot_decode(encoded * 100)  # sharp logits ≈ one-hot
assert torch.allclose(decoded.squeeze(-1), test_vals.squeeze(-1), atol=0.2)

# 3) Loss is finite + backward works
recon_loss = F.mse_loss(obs_pred, symlog(obs_seq))
rew_target = twohot_encode(symlog(rew_seq[:, :-1]))
rew_log_probs = F.log_softmax(rew_logits[:, 1:], dim=-1)
reward_loss = -(rew_target * rew_log_probs).sum(-1).mean()
cont_target = 1.0 - done_seq[:, :-1]
cont_loss = F.binary_cross_entropy_with_logits(cont_logits[:, 1:], cont_target)

kl_for_prior, kl_for_post = [], []
for post, prior in zip(out["post_dists"], out["prior_dists"]):
    post_sg = Normal(post.mean.detach(), post.stddev.detach())
    prior_sg = Normal(prior.mean.detach(), prior.stddev.detach())
    kl_for_prior.append(
        torch.distributions.kl.kl_divergence(post_sg, prior).sum(dim=-1, keepdim=True)
    )
    kl_for_post.append(
        torch.distributions.kl.kl_divergence(post, prior_sg).sum(dim=-1, keepdim=True)
    )
kl_prior = torch.stack(kl_for_prior, dim=1)
kl_post = torch.stack(kl_for_post, dim=1)
kl_loss = (
    CFG.kl_balance * torch.clamp(kl_prior - CFG.free_nats, min=0.0).mean()
    + (1 - CFG.kl_balance) * torch.clamp(kl_post - CFG.free_nats, min=0.0).mean()
)

wm_loss = (
    CFG.recon_scale * recon_loss
    + CFG.reward_scale * reward_loss
    + CFG.cont_scale * cont_loss
    + CFG.kl_scale * kl_loss
)
assert torch.isfinite(wm_loss).all()

wm.zero_grad(set_to_none=True)
wm_loss.backward()
wm_gn = (
    sum(float(p.grad.detach().pow(2).sum()) for p in wm.parameters() if p.grad is not None) ** 0.5
)
print(f"WM sanity: loss={float(wm_loss.item()):.4f}, grad_norm={wm_gn:.6f}")

# 4) Gradient flows into the actor through imagination (no optimizer step)
start = wm.rssm.init_state(B)
feats, actions, entropies, rewards, conts, last_feat = imagine_rollout(start, horizon=5)
assert feats.shape[:2] == (B, 5)
assert rewards.shape == (B, 5, 1)
assert conts.shape == (B, 5, 1)

discounts = CFG.gamma * conts
weights = torch.cumprod(
    torch.cat([torch.ones(B, 1, 1, device=DEVICE), discounts[:, :-1]], dim=1),
    dim=1,
).detach()

values_sg = symexp(twohot_decode(value_net(feats))).detach()
bootstrap = symexp(twohot_decode(value_net(last_feat))).detach()  # [B, 1]

returns = lambda_return(
    rewards=rewards,
    values=values_sg,
    discounts=discounts,
    bootstrap=bootstrap,
    lam=CFG.lam,
)

advantages = returns - values_sg
actor_loss = -(weights * advantages).mean()
actor.zero_grad(set_to_none=True)
actor_loss.backward()

act_gn = (
    sum(float(p.grad.detach().pow(2).sum()) for p in actor.parameters() if p.grad is not None)
    ** 0.5
)
print(f"Actor sanity: actor_loss={float(actor_loss.item()):.4f}, grad_norm={act_gn:.6f}")
assert act_gn > 0.0, "Actor grad norm is zero — check gradient flow through imagination."

print("Sanity checks passed!")

## Data collection

We collect real transitions **one step at a time** and store full episodes in the replay buffer.

Dreamer interleaves data collection and training at the **step level**: after each environment step,
we perform `train_ratio` gradient updates on the world model and the actor/critic. This keeps the
model and policy fresh relative to the data being collected.

**MuJoCo NaN guard:** physics simulators like MuJoCo can produce non-finite observations when the
simulation diverges (e.g., extreme joint velocities after a bad action sequence). A single NaN
observation stored in the replay buffer would poison every training batch that samples it. We check
each observation and discard the episode if non-finite values are detected.


In [ ]:
@torch.no_grad()
def policy_action(obs: np.ndarray, state: RSSMState, prev_action_env: np.ndarray):
    """Select an action using the current policy, updating the RSSM state."""
    obs_t = torch.as_tensor(obs, device=DEVICE).float().unsqueeze(0)
    prev_a = torch.as_tensor(prev_action_env, device=DEVICE).float().unsqueeze(0)

    embed = wm.encoder(obs_t)
    state, post, prior = wm.rssm.obs_step(state, prev_a, embed)

    feat = wm.rssm.features(state)
    a_tanh, _ = actor.sample(feat)  # discard entropy
    a_env = to_action_env(a_tanh)[0]
    return a_env, state

## Train

The main training loop follows Dreamer's **step-level interleaving**:

1. **Seed phase**: collect `seed_steps` random transitions to bootstrap the replay buffer.
2. **Training phase**: for each environment step, collect one transition, then perform `train_ratio`
   world model updates and `train_ratio` actor/critic updates.

When an episode ends, we store it in the replay buffer and immediately start a new one. This is
conceptually different from on-policy methods where we need to wait for a full trajectory before
learning.

You can adjust `CFG.train_steps` for shorter/longer runs. The curve should begin improving after the
seed phase. Hopper can be noisy; if the curve looks flat, try running longer.


In [ ]:
def train_dreamer(env_name: str, **cfg_overrides):
    """Run the full Dreamer training loop on env_name.

    Creates a fresh env, WM, actor, critic, target critic, and replay buffer,
    then runs: seed phase -> WM pretraining -> interleaved env/training with
    periodic eval and best-checkpoint tracking -> restore best checkpoint.

    Any keyword arguments override fields on the global `CFG` dataclass for the
    duration of the run (and persist after — this is intentional so the config
    reflects what was actually used).

    Returns a `history` dict with per-eval metrics.
    """
    # Apply overrides to CFG in place.
    for k, v in cfg_overrides.items():
        if not hasattr(CFG, k):
            raise AttributeError(f"Config has no field {k!r}")
        setattr(CFG, k, v)

    # These are reassigned so helper functions (which reference them as globals)
    # pick up the new env/models for this run.
    global ENV_NAME, env, obs_dim, act_dim, act_low, act_high
    global wm, actor, value_net, value_net_target
    global opt_wm, opt_actor, opt_value, replay

    ENV_NAME = env_name
    env = make_env(SEED)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    act_low = torch.as_tensor(env.action_space.low, device=DEVICE, dtype=torch.float32)
    act_high = torch.as_tensor(env.action_space.high, device=DEVICE, dtype=torch.float32)

    wm = WorldModel(
        obs_dim=obs_dim,
        act_dim=act_dim,
        deter_dim=CFG.deter_dim,
        stoch_dim=CFG.stoch_dim,
        hidden=CFG.hidden_dim,
    ).to(DEVICE)
    feat_dim = wm.rssm.deter_dim + wm.rssm.stoch_dim
    actor = TanhGaussianPolicy(feat_dim=feat_dim, act_dim=act_dim, max_std=CFG.max_std).to(DEVICE)
    value_net = ValueNet(feat_dim=feat_dim).to(DEVICE)
    value_net_target = ValueNet(feat_dim=feat_dim).to(DEVICE)
    value_net_target.load_state_dict(value_net.state_dict())
    for p in value_net_target.parameters():
        p.requires_grad = False
    opt_wm = torch.optim.AdamW(wm.parameters(), lr=CFG.lr_wm, weight_decay=CFG.wm_weight_decay)
    opt_actor = torch.optim.Adam(actor.parameters(), lr=CFG.lr_actor)
    opt_value = torch.optim.Adam(value_net.parameters(), lr=CFG.lr_value)
    replay = ReplayBuffer(capacity_episodes=CFG.capacity_episodes)

    # Reset module-level state from previous runs.
    _return_scale["ema"] = 1.0

    print(f"=== Training Dreamer on {env_name} ===")
    print(f"obs_dim={obs_dim} act_dim={act_dim}")
    print(f"World model: {sum(p.numel() for p in wm.parameters()):,} params")
    print(f"Actor:       {sum(p.numel() for p in actor.parameters()):,} params")
    print(f"Critic:      {sum(p.numel() for p in value_net.parameters()):,} params")

    history = {
        "env_steps": [],
        "eval_mean": [],
        "eval_std": [],
        "wm_loss": [],
        "actor_loss": [],
        "value_loss": [],
    }

    # Best-checkpoint tracking: Dreamer with a small WM can find a good policy
    # and then drift away from it as the WM keeps updating. Snapshotting the
    # best-seen weights and restoring them at the end gives us the peak
    # performance rather than the last-step performance. This is not part of
    # DreamerV3 as published — it's a pragmatic convenience for this notebook.
    best = {
        "eval_mean": -float("inf"),
        "step": -1,
        "wm": None,
        "actor": None,
        "value_net": None,
    }

    def snapshot_to_cpu(module):
        return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

    def sample_batch():
        try:
            return replay.sample_sequences(CFG.batch_size, CFG.seq_len)
        except ValueError:
            return None

    _train_step_count = 0

    def train_step():
        nonlocal _train_step_count
        batch = sample_batch()
        if batch is None:
            return {}
        obs_seq, act_seq, rew_seq, done_seq = batch
        out = {}
        out.update(world_model_update(obs_seq, act_seq, rew_seq, done_seq))
        _train_step_count += 1
        if _train_step_count % CFG.actor_update_every == 0:
            out.update(actor_critic_update(obs_seq, act_seq, rew_seq, done_seq))
        return out

    env_steps = 0
    t0 = time.time()

    # ---- Seed phase: collect random transitions ----
    # `terminated` vs `truncated`: only true termination counts as cont=0 for
    # the continuation head. Time-limit truncation is a boundary but not a
    # death; storing it as death poisons the cont head (important for Pendulum,
    # which ONLY ever truncates).
    print("Collecting seed data...")
    obs, _ = env.reset()
    ep = EpisodeBuffer()
    while env_steps < CFG.seed_steps:
        action = env.action_space.sample()
        next_obs, rew, terminated, truncated, info = env.step(action)
        end = terminated or truncated
        if not np.isfinite(next_obs).all() or not np.isfinite(rew):
            ep = EpisodeBuffer()
            obs, _ = env.reset()
            continue
        ep.add(obs, action, rew, float(terminated))
        env_steps += 1
        obs = next_obs
        if end:
            replay.add_episode(ep)
            ep = EpisodeBuffer()
            obs, _ = env.reset()
    if len(ep) > 1:
        replay.add_episode(ep)
    seed_lens = [len(e) for e in replay.episodes]
    print(
        f"Seed: {len(replay)} episodes, {env_steps} steps. "
        f"len min/median/max={min(seed_lens)}/{int(np.median(seed_lens))}/{max(seed_lens)}, "
        f"eligible(>={CFG.seq_len})={sum(1 for L in seed_lens if L >= CFG.seq_len)}"
    )

    # ---- World-model pretraining phase ----
    # Train ONLY the world model on the seed data for a few hundred gradient
    # steps before the actor starts updating. This prevents the very first
    # actor gradient from flowing through a random, uninformative WM.
    print(f"World model pretraining ({CFG.wm_pretrain_steps} steps)...")
    for i in range(CFG.wm_pretrain_steps):
        batch = sample_batch()
        if batch is None:
            print("  no batch available — aborting pretraining.")
            break
        obs_seq, act_seq, rew_seq, done_seq = batch
        pt_logs = world_model_update(obs_seq, act_seq, rew_seq, done_seq)
        if (i + 1) % 100 == 0:
            print(
                f"  pretrain step {i+1}/{CFG.wm_pretrain_steps}  wm/loss={pt_logs['wm/loss']:.3f}"
            )

    # ---- Main training phase ----
    print("Training...")
    obs, _ = env.reset()
    ep = EpisodeBuffer()
    rssm_state = wm.rssm.init_state(batch_size=1)
    prev_action = np.zeros(act_dim, dtype=np.float32)
    next_eval = env_steps
    logs = {}

    while env_steps < CFG.train_steps:
        action, rssm_state = policy_action(obs, rssm_state, prev_action)
        next_obs, rew, terminated, truncated, info = env.step(action)
        end = terminated or truncated

        if not np.isfinite(next_obs).all() or not np.isfinite(rew):
            ep = EpisodeBuffer()
            next_obs, _ = env.reset()
            rssm_state = wm.rssm.init_state(batch_size=1)
            prev_action = np.zeros(act_dim, dtype=np.float32)
            obs = next_obs
            continue

        ep.add(obs, action, rew, float(terminated))
        env_steps += 1

        if end:
            replay.add_episode(ep)
            ep = EpisodeBuffer()
            next_obs, _ = env.reset()
            rssm_state = wm.rssm.init_state(batch_size=1)
            prev_action = np.zeros(act_dim, dtype=np.float32)
        else:
            prev_action = action

        obs = next_obs

        for _ in range(CFG.train_ratio):
            logs = train_step()

        if env_steps >= next_eval:
            mean_ret, std_ret = eval_policy(ENV_NAME, wm, actor, episodes=CFG.eval_episodes)
            history["env_steps"].append(env_steps)
            history["eval_mean"].append(mean_ret)
            history["eval_std"].append(std_ret)
            history["wm_loss"].append(logs.get("wm/loss", np.nan))
            history["actor_loss"].append(logs.get("ac/actor_loss", np.nan))
            history["value_loss"].append(logs.get("ac/value_loss", np.nan))

            is_best = mean_ret > best["eval_mean"]
            if is_best:
                best["eval_mean"] = mean_ret
                best["step"] = env_steps
                best["wm"] = snapshot_to_cpu(wm)
                best["actor"] = snapshot_to_cpu(actor)
                best["value_net"] = snapshot_to_cpu(value_net)

            ret_m = logs.get("ac/return_mean", float("nan"))
            ent_m = logs.get("ac/entropy", float("nan"))
            elapsed = time.time() - t0
            marker = " *" if is_best else "  "
            # Line 1: eval + losses
            print(
                f"[steps={env_steps:6d}]{marker} eval={mean_ret:8.1f}+/-{std_ret:5.1f}  "
                f'wm={history["wm_loss"][-1]:.3f} al={history["actor_loss"][-1]:.3f} '
                f'vl={history["value_loss"][-1]:.3f} '
                f"imag_ret={ret_m:7.2f} ent={ent_m:.2f}  "
                f"replay={len(replay)}eps  t={elapsed/60:.1f}m"
            )
            # Line 2: diagnostics — WM component breakdown + actor internals
            wm_recon = logs.get("wm/recon", float("nan"))
            wm_rew = logs.get("wm/reward", float("nan"))
            wm_cont = logs.get("wm/cont", float("nan"))
            wm_kl = logs.get("wm/kl_raw", float("nan"))
            ac_std = logs.get("ac/actor_std", float("nan"))
            adv_m = logs.get("ac/adv_mean", float("nan"))
            adv_s = logs.get("ac/adv_std", float("nan"))
            ent_term = logs.get("ac/ent_term", float("nan"))
            ret_scale = logs.get("ac/return_scale", float("nan"))
            print(
                f"         wm: recon={wm_recon:.3f} rew={wm_rew:.3f} "
                f"cont={wm_cont:.3f} kl={wm_kl:.3f}  "
                f"ac: std={ac_std:.3f} adv={adv_m:+.3f}±{adv_s:.3f} "
                f"ent_term={ent_term:+.4f} ret_scale={ret_scale:.2f}"
            )
            next_eval += CFG.eval_every_steps

    if len(ep) > 1:
        replay.add_episode(ep)

    # ---- Restore the best checkpoint ----
    if best["wm"] is not None:
        wm.load_state_dict({k: v.to(DEVICE) for k, v in best["wm"].items()})
        actor.load_state_dict({k: v.to(DEVICE) for k, v in best["actor"].items()})
        value_net.load_state_dict({k: v.to(DEVICE) for k, v in best["value_net"].items()})
        print(
            f"Done. Restored best checkpoint: step={best['step']} "
            f"eval_mean={best['eval_mean']:.1f}"
        )
    else:
        print("Done. No best checkpoint recorded.")

    history["best_step"] = best["step"]
    history["best_eval_mean"] = best["eval_mean"]
    return history

In [ ]:
# Run Dreamer on InvertedPendulum-v5. Random policy episodes on IP are
# very short (~5 steps before the pole tips), so we need a lot of seed
# data AND a short training sequence length to have enough eligible
# episodes for the world model to actually learn on.
history = train_dreamer(
    "InvertedPendulum-v5",
    seed_steps=10_000,
    train_steps=30_000,
    seq_len=8,
    eval_episodes=10,
)

## Results

Let's visualize the training progress: evaluation returns (with standard deviation), world model
loss, and actor/critic losses.


In [ ]:
if len(history["env_steps"]) > 0:
    steps = history["env_steps"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Eval returns with std band, and a marker at the best checkpoint
    mean = np.array(history["eval_mean"])
    std = np.array(history["eval_std"])
    axes[0].plot(steps, mean)
    axes[0].fill_between(steps, mean - std, mean + std, alpha=0.2)
    if history.get("best_step", -1) > 0:
        axes[0].axvline(
            history["best_step"],
            color="red",
            linestyle="--",
            label=f'best={history["best_eval_mean"]:.1f}',
        )
        axes[0].legend()
    axes[0].set_xlabel("Environment steps")
    axes[0].set_ylabel("Eval return")
    axes[0].set_title(f"Dreamer on {ENV_NAME}")
    axes[0].grid(True, alpha=0.3)

    # World model loss
    axes[1].plot(steps, history["wm_loss"])
    axes[1].set_xlabel("Environment steps")
    axes[1].set_ylabel("World model loss")
    axes[1].set_title("World model training")
    axes[1].grid(True, alpha=0.3)

    # Actor/Critic losses
    axes[2].plot(steps, history["actor_loss"], label="actor")
    axes[2].plot(steps, history["value_loss"], label="critic")
    axes[2].set_xlabel("Environment steps")
    axes[2].set_ylabel("Loss")
    axes[2].set_title("Actor & Critic")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
else:
    print("No history recorded (training likely did not run).")

## Rollout

Dreamer's policy is RSSM-stateful: at each step it updates a latent belief `(h, z)` from the full
history of observations. The `DreamerAgent` wrapper below holds this state so it plugs into the
stateless `agent.act(obs)` interface expected by `gym_simulation`.


In [ ]:
from util.gymnastics import gym_simulation


class DreamerAgent:
    """Stateful wrapper: holds RSSM state across steps so the stateless
    `agent.act(obs)` interface expected by `gym_simulation` works with
    Dreamer's history-dependent policy."""

    def __init__(self, wm, actor, act_low, act_high):
        self.wm = wm
        self.actor = actor
        self.act_low = act_low
        self.act_high = act_high
        self.reset()

    def reset(self):
        self.rssm_state = self.wm.rssm.init_state(batch_size=1)
        self.prev_action = torch.zeros(1, self.act_low.shape[0], device=DEVICE)

    @torch.no_grad()
    def act(self, obs):
        obs_t = torch.as_tensor(obs, device=DEVICE).float().unsqueeze(0)
        embed = self.wm.encoder(obs_t)
        self.rssm_state, _post, _prior = self.wm.rssm.obs_step(
            self.rssm_state, self.prev_action, embed
        )
        feat = self.wm.rssm.features(self.rssm_state)
        a_tanh = self.actor.mode(feat)
        a_env = self.act_low + (a_tanh + 1.0) * 0.5 * (self.act_high - self.act_low)
        self.prev_action = a_env
        return a_env.detach().cpu().numpy()[0]

In [ ]:
agent = DreamerAgent(wm, actor, act_low, act_high)
gym_simulation(ENV_NAME, agent=agent, max_t=1000, seed=SEED + 999)

## Optional: Train on Hopper

Now that we've verified the algorithm on InvertedPendulum, we can try the much harder
[Hopper-v5](https://gymnasium.farama.org/environments/mujoco/hopper/) benchmark. Hopper is the
classic Dreamer task: an 11-D observation, 3-D action, dense reward robot that has to learn a
periodic hopping gait. The cell below reuses the exact same `train_dreamer` function as
InvertedPendulum, just with Hopper-tuned hyperparameters:

- **Larger WM** (`deter=256, stoch=64, hidden=256`) — more capacity for the 11-D dynamics.
- **`reward_scale=5.0`** — with 11-D observations, the WM neglects reward prediction unless we
  up-weight it. This was the most impactful tuning discovery in our experiments.
- **`gamma=0.997`** — locomotion payoff is spread over hundreds of steps; a higher discount makes
  the critic value forward motion over mere survival.
- **`actor_update_every=1`** — no throttling; the reward scaling and advantages keep the actor
  grounded without artificial slowdown.
- **`lr_wm=1e-4, grad_clip=100`** — gentler WM optimization, matching DreamerV3 defaults.

**Realistic expectations**: with a 100K-step budget and our educational-scale networks, expect eval
returns in the ~150 range (learned balancing, limited locomotion). See the **Traps and Fragility**
section for why this plateaus and what DreamerV3 does differently.

> Runtime note: 100K Hopper steps takes ~2–4 hours on CPU. If you just want to see the code work,
> try `train_steps=30_000` first.


In [ ]:
history = train_dreamer(
    "Hopper-v5",
    seed_steps=10_000,
    train_steps=50_000,
    wm_pretrain_steps=1000,
    seq_len=16,
    horizon=15,
    entropy_scale=1e-3,
    max_std=0.5,
    eval_every_steps=5000,
    eval_episodes=10,
    lr_wm=1e-4,
    grad_clip=100.0,
    capacity_episodes=5000,
    reward_scale=5.0,
    actor_update_every=1,
    gamma=0.997,
)

In [ ]:
if len(history["env_steps"]) > 0:
    steps = history["env_steps"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    mean = np.array(history["eval_mean"])
    std = np.array(history["eval_std"])
    axes[0].plot(steps, mean)
    axes[0].fill_between(steps, mean - std, mean + std, alpha=0.2)
    if history.get("best_step", -1) > 0:
        axes[0].axvline(
            history["best_step"],
            color="red",
            linestyle="--",
            label=f'best={history["best_eval_mean"]:.1f}',
        )
        axes[0].legend()
    axes[0].set_xlabel("Environment steps")
    axes[0].set_ylabel("Eval return")
    axes[0].set_title(f"Dreamer on {ENV_NAME}")
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(steps, history["wm_loss"])
    axes[1].set_xlabel("Environment steps")
    axes[1].set_ylabel("World model loss")
    axes[1].set_title("World model training")
    axes[1].grid(True, alpha=0.3)
    axes[2].plot(steps, history["actor_loss"], label="actor")
    axes[2].plot(steps, history["value_loss"], label="critic")
    axes[2].set_xlabel("Environment steps")
    axes[2].set_ylabel("Loss")
    axes[2].set_title("Actor & Critic")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

In [ ]:
agent = DreamerAgent(wm, actor, act_low, act_high)
gym_simulation(ENV_NAME, agent=agent, max_t=1000, seed=SEED + 999)

## Traps and Fragility — Things to Watch Out For

Dreamer is one of the most beautiful ideas in modern RL, but in practice it is also one of the most
fragile algorithms in this course. Most of the engineering tricks in this notebook (target critic,
KL balancing, symlog, free nats, return scaling, std floor, AdamW with weight decay) exist because
of failure modes that show up _immediately_ when you scale down or change environments. This section
is a field guide to those traps so that, when you see them, you can recognize and react.

### 1. Sequence-length data starvation (silent training stall)

The world model is trained on contiguous time slices of length `seq_len`. Sampling requires **at
least one stored episode** with `episode_length >= seq_len`. If your environment is one where the
random policy terminates very quickly (e.g. `InvertedPendulum-v5`, where random episodes last ~5
steps), and you set `seq_len=50`, then early in training **zero training steps actually run** — the
sampler keeps returning `None` and the loop silently no-ops.

**Symptoms:** loss values stuck at the initialization, eval flat, no obvious error.<br> **Fix:**
start with a `seq_len` smaller than the typical random-policy episode length. On Hopper random
episodes are tens of steps, so `seq_len=16` is safe. Always log how many of your stored episodes are
eligible for sampling.

### 2. Model exploitation — the imagined-vs-real return gap

Once the actor starts optimizing against the world model, it will _aggressively_ search for weak
spots: states where the model predicts huge reward but the real env doesn't. This is the model-based
RL equivalent of reward hacking, and it is the single biggest reason naive Dreamer implementations
fail. With Gaussian latents and pathwise gradients, the actor can make tiny, precise adjustments
that smoothly exploit world model errors.

**Symptoms:** `imagined_return >> real_return` (e.g. 5–10×), critic value rising monotonically while
eval return stagnates or oscillates wildly.<br> **Mitigations baked into this notebook:**

- **Advantages, not raw returns**: the actor loss uses `returns - values` (centered by the critic
  baseline). When the WM inflates imagined returns, the critic inflates too, so advantages stay
  centered near zero and the actor only responds to genuine improvements.
- **Actor throttling** (`actor_update_every`): the world model can get multiple gradient steps per
  actor update. This keeps the WM ahead of the actor. (Default: 4 for IP; Hopper runs best with 1
  when `reward_scale` and advantages are in place.)
- **Target critic** (Polyak EMA, $\tau=0.02$): the TD($\lambda$) bootstrap uses a slow copy of the
  critic, so the actor cannot make the critic chase its own tail.
- **Return scaling** (EMA of P95–P5 spread): keeps the actor objective scale-invariant, so the
  entropy bonus stays meaningful no matter how big the imagined returns get.
- **Two-hot distributional heads** for reward and value: a distributional head trained with
  cross-entropy is harder to exploit than a scalar MSE head — it learns a sharper, more calibrated
  mapping, and limits how badly the WM can be wrong about any single prediction.
- **Symlog targets** on observations and rewards: compresses outliers and keeps the reconstruction /
  reward heads from being dominated by a few extreme states.
- **AdamW with weight decay** on the world model: small but real regularization against memorizing
  exact replay sequences.
- **Std floor** on the policy ($\sigma \geq 0.1$): the actor cannot collapse to a deterministic
  exploit.

### 3. Continuation head pathology

The continuation head models $p(\text{episode continues})$ and acts as a learned discount inside
imagination. If it learns to always predict `1.0`, imagination rolls on forever and the actor
optimizes a phantom infinite return. If it learns to always predict `0.0`, the imagined horizon
collapses to a single step.

**Symptoms:** weight on later imagination steps near 0 or all near 1; `cont_loss` stuck.<br>
**Fix:** make sure `done=True` is recorded only on _termination_ (early stop), **not** on truncation
(time-limit). MuJoCo + `RecordEpisodeStatistics` handles this correctly, but if you write your own
collector watch this carefully.

### 4. KL collapse / KL explosion

The KL between posterior $q_\theta(z_t \mid h_t, o_t)$ and prior $p_\theta(z_t \mid h_t)$ is what
makes the RSSM useful: the prior has to be predictable from the posterior, otherwise imagination is
meaningless.

- **KL collapse:** posterior ignores observations, imagination becomes garbage.
- **KL explosion:** posterior overfits each observation, prior has no idea, imagination again
  becomes garbage.

**Mitigations baked in:**

- **KL balancing** (DreamerV2): split the KL gradient 80% on the prior, 20% on the posterior, so the
  prior learns to chase the posterior.
- **Free nats**: don't penalize the KL below a small floor, so early training can use the posterior
  freely without being clamped to the (still untrained) prior.

### 5. "I changed environments and everything broke"

Most of the hyperparameters in `Config` are **environment-specific**. Switching from Hopper to a
continuing-task env like Pendulum, a sparse-reward env, or a pixel env will likely require
re-tuning. Things that change with the environment:

- `seq_len` — must be shorter than typical random episodes
- `seed_steps` — needs more if random episodes are short
- `entropy_scale` — what's "exploration" depends on action dim and reward scale
- `reward_scale` — must increase when obs_dim >> 1 to prevent reconstruction from drowning out
  reward
- `gamma` — tasks with long-horizon payoff (locomotion) need higher gamma (0.997 vs 0.99)
- `horizon` — on stiff envs like Hopper, longer imagination amplifies model error
- WM hidden sizes — bigger envs need more capacity, but bigger WMs also exploit more

### 6. Compute scale gap from the paper

The published DreamerV3 results on Hopper use roughly **1M env steps**, larger networks
(`hidden=512+`, `deter=512+`), discrete categorical latents, pixel observations through a CNN, and
observation normalization. This notebook strips most of those away for clarity. The educational
point is _the algorithmic loop_ (RSSM → imagine → backprop through dynamics into the actor), not
state-of-the-art Hopper scores. Treat the training curves you'll see below accordingly: if you want
to chase paper numbers, scale up.

### 7. Things that look right but aren't, in a Dreamer context

- **Advantage normalization in the actor loss.** It's tempting (we all do it for PPO), but Dreamer's
  actor uses _pathwise_ gradients through the dynamics — not REINFORCE — and rescaling the advantage
  distorts those gradients in a way that hurts. Use return scaling via EMA of P95–P5 spread instead,
  and leave the per-batch return distribution alone.
- **Increasing `train_ratio` blindly.** More gradient updates per env step accelerates model
  exploitation as much as it accelerates learning. Bumping it can help, but only after the other
  stabilizers are in place.
- **Bigger world model = better.** Up to a point — but a bigger WM is also a more powerful fantasy
  machine for the actor to exploit. If you scale WM size, scale critic stabilization (target $\tau$,
  return scaling) along with it.

### 8. Loss imbalance — the WM learns what you tell it to

This was the most impactful discovery in our Hopper experiments. With all loss scales at 1.0, the
world model will prioritize whichever prediction head has the largest loss. For Hopper (11-D
observations, 1-D reward), the reconstruction loss gradient from 11 dimensions drowns out the reward
head's gradient from 1 dimension. The WM learns to reconstruct observations beautifully (recon loss
→ 0.006) while reward prediction barely improves (reward loss stuck at 0.50+).

Since the actor optimizes imagined _returns_, bad reward prediction means bad actor gradients —
regardless of how perfect the observation predictions are. Setting `reward_scale=5.0` for Hopper
rebalances the gradient and was the single biggest improvement in our tuning experiments.

**Diagnostic recipe:** if your agent plateaus, log the WM component losses separately (`wm/recon`,
`wm/reward`, `wm/cont`, `wm/kl`). If one head is 100× worse than the others, up-weight it. This
notebook includes diagnostic logging for exactly this purpose.

### 9. Discount factor and temporal credit assignment

On tasks where the payoff from good behavior is spread over many future steps (e.g., locomotion in
Hopper: forward velocity rewards accumulate over hundreds of steps), the discount factor $\gamma$
matters more than you'd think. With $\gamma = 0.99$ (effective horizon ~100 steps), the critic
mostly values near-term survival. With $\gamma = 0.997$ (effective horizon ~333 steps), the critic
can value locomotion over standing still. For Hopper, $\gamma = 0.997$ (DreamerV3's default)
noticeably improved training stability.


## Conclusion & Next Steps

In this notebook we built a Dreamer-style agent from scratch and trained it on MuJoCo tasks. Let's
recap the key ideas:

- The **RSSM** combines deterministic (GRU) and stochastic (Gaussian) state to model environment
  dynamics in a compact latent space.
- The **world model** is trained on real sequences with reconstruction, reward, continuation, and KL
  losses — using **symlog** targets, **two-hot distributional heads** for reward and value, and **KL
  balancing** to prevent posterior/prior collapse.
- The **actor and critic** are trained entirely on **imagined trajectories** — the agent literally
  learns by dreaming.
- **Gradients flow through the model dynamics** into the actor, making this a true
  backpropagation-through-time approach (unlike REINFORCE-style model-free methods).
- A **target critic** (Polyak EMA) stabilizes the TD($\lambda$) bootstrap, and **return scaling**
  keeps the actor objective scale-invariant so the entropy bonus stays meaningful.
- **Loss balancing** (`reward_scale`) is critical when observation dimensionality exceeds reward
  dimensionality — without it, the WM neglects reward prediction and the actor optimizes noise.

Compared to MBPO (from the [MBRL notebook](./15_MBRL.ipynb)), Dreamer doesn't need a separate
model-free agent or state-space rollouts — everything happens in latent space, and the model is used
as a _differentiable simulator_ rather than a data generator.

**Realistic expectations.** With our training budgets you should see returns climb out of
random-policy territory. InvertedPendulum should reach 700+ within 30K steps, and Hopper should show
clear learning (eval ~150, learned balancing) within 100K steps. That is far from the 1000+ scores
DreamerV3 reports on Hopper — those need ~1M env steps, `train_ratio=512`, larger networks,
categorical latents, and observation normalization. The
[Traps and Fragility](#Traps-and-Fragility-—-Things-to-Watch-Out-For) section above explains the
failure modes you'll hit if you try to push further without scaling up first.

### How this differs from full DreamerV3

This notebook is a simplified, educational version. Key differences from the
[DreamerV3 paper](https://arxiv.org/abs/2301.04104):

- **Latents**: we use Gaussian continuous latents; DreamerV3 uses discrete categorical latents
  (which make the latent space harder to exploit via pathwise gradients at scale).
- **Observations**: we use vector state; DreamerV3 is designed for pixels (CNN encoder/decoder).
- **Train ratio**: we use `train_ratio=1` (one gradient step per env step); DreamerV3 uses 512. This
  is the single biggest gap — at ratio 512 the WM is essentially converged on the current replay
  distribution before the actor ever updates, eliminating most exploitation.
- **Scale**: DreamerV3 uses much larger networks (512+ hidden) and many more env steps (~1M+), plus
  dozens of small engineering details (observation normalization, unimix categoricals, etc.) we omit
  for clarity.

### Exercises

1. **Scale up the Hopper run.** Increase `train_steps` to 500K, wrap the env in
   `gymnasium.wrappers.NormalizeObservation`, and bump WM hidden sizes to 512. Compare the training
   curves to our shorter run.
2. **Try a friendlier env.** Switch `ENV_NAME` to `Pendulum-v1` (continuous classic control) and
   watch Dreamer learn cleanly within a small budget. This is the easiest way to confirm the
   algorithm itself is correct.
3. **Switch to pixel observations**: add a CNN encoder and transposed-CNN decoder.
4. **Replace Gaussian latents with discrete categorical** latents (as in DreamerV2/V3). Use the
   straight-through estimator for gradient flow and see if it helps on Hopper at scale.
5. **Experiment with `train_ratio`**: what happens with 2, 4, or 8 gradient updates per env step?
   Watch the imagined-vs-real return gap as you crank it up — that's model exploitation in action.
6. **Tune `reward_scale`**: try values from 1 to 20 and watch how `wm/reward` loss changes in the
   diagnostic output. What happens to eval returns as reward prediction improves?
